## 一、背景介绍

### 1.1 医学科普

* 眼底荧光血管造影（FFA）技术解析与临床应用

眼底荧光血管造影（Fundus Fluorescein Angiography, FFA）是眼科领域重要的诊断技术，属于微创性检查手段。其操作原理为：通过肘部静脉注射荧光素钠造影剂，利用特殊滤光片的眼底相机连续拍摄，动态观察造影剂在视网膜血管及黄斑区循环的充盈过程，从而精准识别血管异常、渗漏、新生血管形成等病理改变，为糖尿病视网膜病变、黄斑变性、视网膜静脉阻塞等疾病的诊断、分型及治疗方案制定提供关键依据。

作为侵入式检查，FFA需遵循严格的操作规范。检查前需评估患者过敏史及肝肾功能，避免造影剂过敏或代谢负担。注射后约10-15秒，造影剂经血液循环抵达眼底，相机启动快速序列拍摄，全程持续5-10分钟，重点捕捉动脉期、静脉期等不同时相的血管形态。通过计算机图像处理技术，可量化分析血管渗漏程度、缺血区域范围等参数，实现从定性到定量的诊断升级。

相较于OCT、B超等无创检查，FFA具有独特的血管动态成像优势，能清晰显示微血管瘤、毛细血管无灌注区等微细结构。但需注意其局限性：部分患者可能出现恶心、皮疹等轻度过敏反应，极少数发生严重过敏反应需紧急处理；孕妇、肾功能不全者需慎用。

随着技术发展，FFA已与多模态影像融合，如结合OCT血管成像（OCTA）实现结构与功能的双重评估。临床研究表明，FFA在指导抗VEGF药物治疗、激光光凝时机选择等方面具有不可替代的价值，是眼科精准医疗的重要基石，持续推动着眼底病诊疗水平的提升。


### 1.2 数据说明

上文提及造影图像，大概如示例：

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/c54a4e08bcbf43858e78fd90672b51303d12d76fc19440c29bd7845dcdfa5d8c" width="400">
</div>

我们需要根据影像，去预测不同的血管形态、异常和血流状态，包括：

| 英文缩写/字段 | 中文全称 | 备注 |
|--------------|----------|------|
| HyperF_Type | 高荧光类型 |  |
| HyperF_Area (DA) | 高荧光-视盘面积 | DA = Disc Area |
| HyperF_Fovea | 黄斑区高荧光 |  |
| HyperF_ExtraFovea | 黄斑外高荧光 |  |
| HyperF_Y | 高荧光纵轴位置 | Y = 垂直方向坐标 |
| HypoF_Type | 低荧光类型 |  |
| HypoF_Area (DA) | 低荧光-视盘面积 | DA = Disc Area |
| HypoF_Fovea | 黄斑区低荧光 |  |
| HypoF_ExtraFovea | 黄斑外低荧光 |  |
| HypoF_Y | 低荧光纵轴位置 | Y = 垂直方向坐标 |
| CNV | 脉络膜新生血管 | Choroidal Neovascularization |
| Vascular abnormality (DR) | 血管异常 | 糖尿病视网膜病变特征 |
| Pattern | 特殊荧光特殊类型 |  |


## 二、技术方案

### 2.1 **Resnet152 图像分类**

基于Resnet152

```python
self.resnet = pdl.nn.Sequential(
    pdl.vision.models.resnet152(num_classes=512, pretrained=True)
)
```

构建图像分类模型，

```python
class PaiPai(pdl.nn.Layer):
	def __init__(self, _N=32 * yN):
		super(PaiPai, self).__init__()
		self._N = _N

		self.model1 = RES()
		self.model1.set_state_dict(pdl.load("./modl/_s4.9997_t5.6384_v4.9997.mdl"))

		self.model2 = pdl.nn.Sequential(
			pdl.nn.Linear(in_features=_x2N, out_features=self._N),
			pdl.nn.ReLU(),

			pdl.nn.Linear(in_features=self._N, out_features=self._N),
			pdl.nn.ReLU(),

			pdl.nn.Linear(in_features=self._N, out_features=self._N),
		)

		self.b0 = self.create_parameter(
			[1], is_bias=True, default_initializer=pdl.nn.initializer.Constant(value=0.0))
		self.w1 = self.create_parameter(
			[1], is_bias=True, default_initializer=pdl.nn.initializer.Constant(value=1.0))
		self.w2 = self.create_parameter(
			[1], is_bias=True, default_initializer=pdl.nn.initializer.Constant(value=0.0))

	def forward(self, _x1, _x2):
		return self.b0 + self.w1 * self.model1(_x1) + self.w2 * self.model2(_x2)
```


### 2.2 **ERNIE-5.0-Thinking-Preview 多模态**

* **系统提示词**

**并通过image_url，传入"data:image/jpeg;base64" 图片。**


In [ ]:
iMessages = lambda _i, _k: [
	{
		"role": "user",
		"content": "你是一个眼科医生",
	}, 
	{
		"role": "assistant",
		"content": "是的，我现在扮演的是眼科医生的角色。根据您提供的眼部信息，我可以给您一些建议，帮助您更好地了解您的眼部健康状况。",
	}, 

	{
		"role": "user",
		"content": "我们的诊断信息主要来自于眼底萤光血管造影，它是一种侵入式的检查，是由手臂静脉注入萤光剂，再由眼底相机照出黄斑部及视网膜血管的变异情形，可作为医师诊断及后续治疗的参考辅助。",
	}, 
	{
		"role": "assistant",
		"content": "具体来说，诊断信息包含哪些内容，我可以知道么？",
	}, 

	{
		"role": "user",
		"content": """
			Data of each eye contains the following information
			每只眼睛的治疗前数据包含以下信息

			ID: patient ID
			
			Folder
			文件夹(眼睛)
			
			Impression of each eye
			每只眼睛的诊断结果
			
			HyperF_Type: hyperfluorescence type
			高荧光类型

			HyperF_Area (DA)
			-4 (The lesion is 4 or smaller than 4 disc areas)
			≤ 4个视盘面积
			-5 (The lesion is larger than 4 disc areas)
			＞4个视盘面积

			HyperF_Fovea: hyperfluorescence (fovea)
			黄斑区高荧光
			
			HyperF_ExtraFovea: hyperfluorescence (extra fovea)
			黄斑外高荧光
			
			HyperF_Y: hyperfluorescence (Y location)
			高荧光纵轴位置
			
			HypoF_Type: hypofluorescence type
			低荧光类
			
			HypoF_Area (DA)
			-4 (The lesion is 4 or smaller than 4 disc areas)
			≤ 4个视盘面积
			-5 (The lesion is larger than 4 disc areas)
			＞4个视盘面积
			
			HypoF_Fovea: hypofluorescence (fovea)
			黄斑区低荧光
			
			HypoF_ExtraFovea: hypofluorescence (extra fovea)
			黄斑外低荧光
			
			HypoF_Y: hypofluorescence (Y location)
			低荧光纵轴位置
			
			CNV: choroidal neovascularization
			脉络膜新生血管
			
			Vascular abnormality (DR)
			血管异常（糖尿病视网膜病变特征）
			
			Pattern
			特殊荧光特殊类型

			Angiography images: including FFA (fundus fluorescein angiography) and/or ICGA (indocyanine green angiography) images. There are multiple images taken based on examination needs.
			造影图像：包括眼底荧光血管造影和/或吲哚菁绿血管造影
		""",
	}, 
	{
		"role": "assistant",
		"content": "好的，我知道了。",
	}, 

    # # 纯文本
	# {
	# 	"role": "user",
	# 	"content": f"{_k} 请根据这些内容，整合成一份眼科疾病诊断报告",
	# },

    # 多模态
    {
		"role": "user",
        "content": [
            {
                "type": "text", 
                "text": f"{_k} 请根据这些内容，整合成一份眼科疾病诊断报告"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{_i}"
                }
            }
        ]
    	},
]

## 三、应用运行

### 3.1 初始化

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/6045bb078858499cb349024c5314cad26c596d50657842cfbba66bc07d3e6a5e" width="300">
</div>

### 3.2 进入页面

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/e4dee65f489549cfbeb7b2ee4dd927e8b5f3c0e9cf2d4d90923d8bdbe48da58e" width="300">
</div>

### 3.3 关于我们

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/36917e53426040e181aa4e577c41b6efbd9335b034bb451ab0c81b573ad9bc3c" width="300">
</div>

### 3.4 提交示例

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/8f433a2f963b4d2c9d1612f57d84c218303fd949f316477c9116745abd6071d0" width="300">
</div>

### 3.5 开始诊断

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/582911b344ef4c778aa12cb6e3f1701d18fd5df8b7ae4863b41250e356c5518c" width="300">
</div>


### 3.6 报告输出

<div>
    <img src="https://ai-studio-static-online.cdn.bcebos.com/9a160d7fdb614c70acc762f31cef9115248b39062ffb499aa6b3ba4b84a8e02e" width="300">
    <img src="https://ai-studio-static-online.cdn.bcebos.com/f6138f9bf1794b8a99dda498495b46e260f33156813242e2aaa61733c0fbf1b0" width="300">
    <img src="https://ai-studio-static-online.cdn.bcebos.com/55192c6d9c984f329095f2ab242ca527f967af22cd1f4112be36d7e098c99b8f" width="300">
</div>


## 四、代码逻辑

### 4.1 环境配置

我们把这部分环境配置，放置到Gradio部署服务，这样的好处是。

特别的，涉及相关代码库更新较为频繁，在此会限制版本。


In [5]:
# ! pip install erniebot==0.5.0
# ! pip install gradio==4.26.0
# ! pip install dotenv==1.2.1
# ! pip install openai==2.14.0

### 4.2 文件结构

```python
main.gradio.py
modl
├── _s4.9997_t5.6384_v4.9997.mdl
└── _s5.5066_t5.4835_v5.5066.mdl
paras.py
work
├── 0.jpg
├── ...
└── working.jpg
```

### 4.3 模型文件

In [1]:
! ls -l modl/*

-rw-r--r-- 1 aistudio aistudio 233639524 Dec 29  2024 modl/_s4.9997_t5.6384_v4.9997.mdl
-rw-r--r-- 1 aistudio aistudio 248995291 Dec 29  2024 modl/_s5.5066_t5.4835_v5.5066.mdl


### 4.4 运行配置

In [2]:
import pickle
import numpy as np
import paddle as pdl
import cv2 as cv

yN = 14

COLS = [
	'Impression',
	'HyperF_Type',
	'HyperF_Area(DA)',
	'HyperF_Fovea',
	'HyperF_ExtraFovea',
	'HyperF_Y',
	'HypoF_Type',
	'HypoF_Area(DA)',
	'HypoF_Fovea',
	'HypoF_ExtraFovea',
	'HypoF_Y',
	'CNV',
	'Vascular abnormality (DR)',
	'Pattern'
]

Mcols = [
	'Impression',
	'HyperF_ExtraFovea',
	'HyperF_Y',
	'HypoF_ExtraFovea',
	'HypoF_Y',
	'Vascular abnormality (DR)',
	'Pattern',
]

McolsN = {
	'Impression': 8,
	'HyperF_ExtraFovea': 8,
	'HyperF_Y': 5,
	'HypoF_ExtraFovea': 8,
	'HypoF_Y': 5,
	'Vascular abnormality (DR)': 7,
	'Pattern': 7,
}

_k2v = {
	'Impression': {
		'macular neovascularization': 0,
		'unremarkable changes': 1,
		'dry age-related macular degeneration': 2,
		'central serous chorioretinopathy': 3,
		'uveitis': 4,
		'chorioretinal scar': 5,
		'diabetic retinopathy': 6,
		'retinal pigment epithelial detachment': 7,
		'polypoidal choroidal vasculopathy': 8,
		'pachychoroid pigment epitheliopathy': 9,
		'chorioretinal atrophy': 10,
		'myopia': 11,
		'cystoid macular edema': 12,
		'choroidal mass': 13,
		'epiretinal membrane': 14,
		'retinal vein occlusion': 15,
		'other': 16,
		'retinal arterial macroaneurysm': 17,
		'proliferative diabetic retinopathy': 18,
		'branch retinal vein occlusion': 19,
		'retinal dystrophy': 20,
		'central retinal vein occlusion': 21,
		'central retinal artery occlusion': 22
	},
	'HyperF_Type': {
		'leakage': 0,
		'no': 1,
		'staining': 2,
		'pooling': 3,
		'window defect': 4
	},
	'HyperF_Area(DA)': {
		'4': 0,
		'no': 1,
		'5': 2
	},
	'HyperF_Fovea': {
		'no': 0,
		'yes': 1
	},
	'HyperF_ExtraFovea': {
		'no': 0,
		'superior': 1,
		'diffuse': 2,
		'temporal': 3,
		'inferior': 4,
		'nasal': 5,
		'superior nasal': 6,
		'superior temporal': 7,
		'inferior temporal': 8,
		'inferior nasal': 9,
		'disc': 10,
		'superior to disc': 11,
		'nasal to disc': 12,
		'inferior to disc': 13,
		'temporal to disc': 14,
		'periphery': 15,
		'venular': 16,
		'superotemporal': 17
	},
	'HyperF_Y': {
		'subretinal': 0,
		'no': 1,
		'intraretinal': 2,
		'preretinal': 3
	},
	'HypoF_Type': {
		'no': 0,
		'blockage': 1,
		'capillary non-perfusion': 2
	},
	'HypoF_Area(DA)': {
		'no': 0,
		'4': 1,
		'5': 2
	},
	'HypoF_Fovea': {
		'no': 0,
		'yes': 1
	},
	'HypoF_ExtraFovea': {
		'no': 0,
		'inferior': 1,
		'diffuse': 2,
		'superior': 3,
		'temporal': 4,
		'nasal': 5,
		'inferior temporal': 6,
		'superior nasal': 7,
		'inferior nasal': 8,
		'superior temporal': 9,
		'superior to disc': 10,
		'disc': 11,
		'nan': 12,
		'temporal to disc': 13,
		'periphery': 14,
		'inferior to disc': 15,
		'nasal to disc': 16
	},
	'HypoF_Y': {
		'no': 0,
		'subretinal': 1,
		'intraretinal': 2,
		'preretinal': 3,
		'nan': 4
	},
	'CNV': {
		'no': 0,
		'yes': 1
	},
	'Vascular abnormality (DR)': {
		'no': 0,
		'microaneurysm': 1,
		'tortuous': 2,
		'intraretinal microvascular abnormalities': 3,
		'retinal neovascularization elsewhere': 4,
		'macroaneurysm': 5,
		'vessel dilation': 6,
		'telangiectasia': 7,
		'collateral vessel': 8,
		'venous beading': 9,
		'retinal neovascularization of the disc': 10,
		'optociliary shunt': 11,
		'vasculitis': 12,
		'mild tortuous vessel': 13,
		'tortuous dilate': 14
	},
	'Pattern': {
		'no': 0,
		'pcv': 1,
		'polyp': 2,
		'ink blot': 3,
		'petaloid': 4,
		'drusen': 5,
		'smoke stack': 6,
		'panretinal photocoagulation': 7,
		'light bulb': 8,
		'starry sky': 9,
		'retinal pigment epithelial tear': 10,
		'laser scar': 11,
		'branching neovascular network': 12,
		'segmental panretinal photocoagulation': 13
	}
}

_k2v_num = {
	_k1: len(_v1)
	for _k1, _v1 in _k2v.items()
}

_v2k = {
	_k1: {_v2: _k2 for _k2, _v2 in _v1.items()}
	for _k1, _v1 in _k2v.items()
}

#
fnormalize = lambda x, _mean=70.132575, _std=34.550730: (x - _mean) / _std
fill32 = lambda x: np.array([1 if i == x else 0 for i in range(32)])
_x2N = 11

class iDataset(pdl.io.Dataset):
	def __init__(self, data_name, data_list):
		self.data_name = data_name
		self.data_list = data_list
		self.D = 40
		self.L, self.R = 384//2, 384 + 384//2

	def __getitem__(self, index):
		_data = self.data_list[index]
		
		img0 = _data["X"]
		img_b, img_g, img_r = cv.split(img0)
		img_m1 = img0[self.L-self.D:self.L+self.D, self.L-self.D:self.L+self.D, ]
		img_m2 = img0[self.L-self.D:self.L+self.D, self.R-self.D:self.R+self.D, ]

		_x1 = fnormalize(_data["X"]).reshape((3, 768, 384)).astype(np.float32)
		_x2 = np.array([
			img0.max(),
			np.percentile(img0, 10),
			img_b.max(),
			img_g.max(),
			img_r.max(),

			img_m1.mean(), img_m1.max(), img_m1.min(),
			img_m2.mean(), img_m2.max(), img_m2.min(),
		], dtype=np.float32)
		
		_y = np.concatenate([
			fill32(_data["Y"]["Impression"]),
			fill32(_data["Y"]["HyperF_Type"]),
			fill32(_data["Y"]["HyperF_Area(DA)"]),
			fill32(_data["Y"]["HyperF_Fovea"]),
			fill32(_data["Y"]["HyperF_ExtraFovea"]),
			fill32(_data["Y"]["HyperF_Y"]),
			fill32(_data["Y"]["HypoF_Type"]),
			fill32(_data["Y"]["HypoF_Area(DA)"]),
			fill32(_data["Y"]["HypoF_Fovea"]),
			fill32(_data["Y"]["HypoF_ExtraFovea"]),
			fill32(_data["Y"]["HypoF_Y"]),
			fill32(_data["Y"]["CNV"]),
			fill32(_data["Y"]["Vascular abnormality (DR)"]),
			fill32(_data["Y"]["Pattern"]),
		], dtype=np.float32)
		return _x1, _x2, _y

	def __len__(self):
		return len(self.data_list)

def jDataset(_data):
	selfD = 40
	selfL, selfR = 384//2, 384 + 384//2

	img0 = _data
	img_b, img_g, img_r = cv.split(img0)
	img_m1 = img0[selfL-selfD:selfL+selfD, selfL-selfD:selfL+selfD, ]
	img_m2 = img0[selfL-selfD:selfL+selfD, selfR-selfD:selfR+selfD, ]

	_x1 = fnormalize(_data).reshape((3, 768, 384)).astype(np.float32)
	_x2 = np.array([
		img0.max(),
		np.percentile(img0, 10),
		img_b.max(),
		img_g.max(),
		img_r.max(),

		img_m1.mean(), img_m1.max(), img_m1.min(),
		img_m2.mean(), img_m2.max(), img_m2.min(),
	], dtype=np.float32)
	return pdl.Tensor(np.array([_x1])), pdl.Tensor(np.array([_x2]))

def model():
	return pdl.nn.Sequential(
		pdl.nn.Linear(in_features=512, out_features=256),
		pdl.nn.ReLU(),

		pdl.nn.Linear(in_features=256, out_features=128),
		pdl.nn.ReLU(),

		pdl.nn.Linear(in_features=128, out_features=64),
		pdl.nn.ReLU(),

		pdl.nn.Linear(in_features=64, out_features=32),
		pdl.nn.ReLU(),

		pdl.nn.Linear(in_features=32, out_features=32),
		pdl.nn.ReLU(),

		pdl.nn.Linear(in_features=32, out_features=32),
	)


class RES(pdl.nn.Layer):
	def __init__(self):
		super(RES, self).__init__()
		self.resnet = pdl.nn.Sequential(
			pdl.vision.models.resnet152(num_classes=512, pretrained=True)
		)

		self.model1 = model()
		self.model2 = model()
		self.model3 = model()
		self.model4 = model()
		self.model5 = model()
		self.model6 = model()
		self.model7 = model()
		self.model8 = model()
		self.model9 = model()
		self.modelA = model()
		self.modelB = model()
		self.modelC = model()
		self.modelD = model()
		self.modelE = model()

	def forward(self, _x):
		_res = self.resnet(_x)
		return pdl.concat([
			self.model1(_res),
			self.model2(_res),
			self.model3(_res),
			self.model4(_res),
			self.model5(_res),
			self.model6(_res),
			self.model7(_res),
			self.model8(_res),
			self.model9(_res),
			self.modelA(_res),
			self.modelB(_res),
			self.modelC(_res),
			self.modelD(_res),
			self.modelE(_res),
		], axis=1)

class PaiPai(pdl.nn.Layer):
	def __init__(self, _N=32 * yN):
		super(PaiPai, self).__init__()
		self._N = _N

		self.model1 = RES()
		self.model1.set_state_dict(pdl.load("./modl/_s4.9997_t5.6384_v4.9997.mdl"))

		self.model2 = pdl.nn.Sequential(
			pdl.nn.Linear(in_features=_x2N, out_features=self._N),
			pdl.nn.ReLU(),

			pdl.nn.Linear(in_features=self._N, out_features=self._N),
			pdl.nn.ReLU(),

			pdl.nn.Linear(in_features=self._N, out_features=self._N),
		)

		self.b0 = self.create_parameter(
			[1], is_bias=True, default_initializer=pdl.nn.initializer.Constant(value=0.0))
		self.w1 = self.create_parameter(
			[1], is_bias=True, default_initializer=pdl.nn.initializer.Constant(value=1.0))
		self.w2 = self.create_parameter(
			[1], is_bias=True, default_initializer=pdl.nn.initializer.Constant(value=0.0))

	def forward(self, _x1, _x2):
		return self.b0 + self.w1 * self.model1(_x1) + self.w2 * self.model2(_x2)


iMessages = lambda _i, _k: [
	{
		"role": "user",
		"content": "你是一个眼科医生",
	}, 
	{
		"role": "assistant",
		"content": "是的，我现在扮演的是眼科医生的角色。根据您提供的眼部信息，我可以给您一些建议，帮助您更好地了解您的眼部健康状况。",
	}, 

	{
		"role": "user",
		"content": "我们的诊断信息主要来自于眼底萤光血管造影，它是一种侵入式的检查，是由手臂静脉注入萤光剂，再由眼底相机照出黄斑部及视网膜血管的变异情形，可作为医师诊断及后续治疗的参考辅助。",
	}, 
	{
		"role": "assistant",
		"content": "具体来说，诊断信息包含哪些内容，我可以知道么？",
	}, 

	{
		"role": "user",
		"content": """
			Data of each eye contains the following information
			每只眼睛的治疗前数据包含以下信息

			ID: patient ID
			
			Folder
			文件夹(眼睛)
			
			Impression of each eye
			每只眼睛的诊断结果
			
			HyperF_Type: hyperfluorescence type
			高荧光类型

			HyperF_Area (DA)
			-4 (The lesion is 4 or smaller than 4 disc areas)
			≤ 4个视盘面积
			-5 (The lesion is larger than 4 disc areas)
			＞4个视盘面积

			HyperF_Fovea: hyperfluorescence (fovea)
			黄斑区高荧光
			
			HyperF_ExtraFovea: hyperfluorescence (extra fovea)
			黄斑外高荧光
			
			HyperF_Y: hyperfluorescence (Y location)
			高荧光纵轴位置
			
			HypoF_Type: hypofluorescence type
			低荧光类
			
			HypoF_Area (DA)
			-4 (The lesion is 4 or smaller than 4 disc areas)
			≤ 4个视盘面积
			-5 (The lesion is larger than 4 disc areas)
			＞4个视盘面积
			
			HypoF_Fovea: hypofluorescence (fovea)
			黄斑区低荧光
			
			HypoF_ExtraFovea: hypofluorescence (extra fovea)
			黄斑外低荧光
			
			HypoF_Y: hypofluorescence (Y location)
			低荧光纵轴位置
			
			CNV: choroidal neovascularization
			脉络膜新生血管
			
			Vascular abnormality (DR)
			血管异常（糖尿病视网膜病变特征）
			
			Pattern
			特殊荧光特殊类型

			Angiography images: including FFA (fundus fluorescein angiography) and/or ICGA (indocyanine green angiography) images. There are multiple images taken based on examination needs.
			造影图像：包括眼底荧光血管造影和/或吲哚菁绿血管造影
		""",
	}, 
	{
		"role": "assistant",
		"content": "好的，我知道了。",
	}, 

    # # 纯文本
	# {
	# 	"role": "user",
	# 	"content": f"{_k} 请根据这些内容，整合成一份眼科疾病诊断报告",
	# },

    # 多模态
    {
		"role": "user",
        "content": [
            {
                "type": "text", 
                "text": f"{_k} 请根据这些内容，整合成一份眼科疾病诊断报告"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{_i}"
                }
            }
        ]
    	},
]

# 

### 4.5 Gradio页面

包括Gradio页面逻辑、应用运作实现。


In [ ]:

import os
os.system("pip install erniebot==0.5.0")
os.system("pip install gradio==4.26.0")
os.system("pip install dotenv==1.2.1")
os.system("pip install openai==2.14.0")

import base64
import uuid
import time
import pickle
import random
import warnings
import datetime
import cv2 as cv
import numpy as np
import pandas as pd
import paddle as pdl
import gradio as gr
from openai import OpenAI

# 通过.env配置
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")

from paras import PaiPai, iDataset, jDataset, COLS, _v2k, yN, _k2v_num, Mcols, McolsN, iMessages

# python3 -c "import uuid;print(str(uuid.uuid1()))"
UUID = str(uuid.uuid1())

SEED = 1024
pdl.seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

encoder = PaiPai()
encoder.set_state_dict(pdl.load("./modl/_s5.5066_t5.4835_v5.5066.mdl"))
encoder.eval()

client = OpenAI(
    api_key=API_KEY, 
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",
)

wrkp = "./work"

_HEAD = """<p align="center"><b>EDA, 眼部疾病诊断AI助手</b></p>"""
_ING1 = """<p align="center" style="color: red;font-weight: bold;">EDA诊断中, 请稍等...</p>"""
_ING2 = """<p><i>眼底萤光血管造影 (FFA) 是由手臂静脉注入萤光剂, 通过眼底相机照出黄斑部及视网膜血管的变异情形, 可作为医师诊断及后续治疗的参考辅助, 也能让文心化身AI眼科医生。</i></p>"""
_ENDS = """<p align="center" style="color: green;">EDA诊断结束</p>"""


_NC, _P, _M, _E = {"C": 0}, {}, {}, {}


getDT = lambda: datetime.datetime.strftime(datetime.datetime.now(), '%Y%m%d-%H%M%S')


def threshold(img):
    try:
        imgray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)  
        ret, thresh = cv.threshold(imgray, 127, 255, cv.THRESH_BINARY)  
        contours, hierarchy = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
        return cv.drawContours(img, contours, -1, (0, 255, 0), 1)
    except:
        return img


def fmodel(x, progress=gr.Progress()):
    for _task in progress.tqdm(range(4)):
        if _task == 0:
            _k = np.sum(x)

        if _task == 1:
            if _k not in _P:
                img = x.copy()
                if img.shape == (434, 768, 3):
                    img = img[:384,:,:,].copy()
                elif img.shape == (409, 384, 3):
                    img.resize((434, 384, 3))
                    img = cv.hconcat([img, img])
                    img = img[:384,:,:,].copy()
                elif img.shape == (434, 384, 3):
                    img.resize((434, 384, 3))
                    img = cv.hconcat([img, img])
                    img = img[:384,:,:,].copy()
                elif img.shape == (306, 512, 3):
                    _img1 = img[:,:256,:].copy()
                    _img2 = img[:,256:,:].copy()
                    _img1 = cv.resize(_img1[:256,:,:,].copy(), (384, 384))
                    _img2 = cv.resize(_img2[:256,:,:,].copy(), (384, 384))
                    img = cv.hconcat([_img1, _img2])
                elif img.shape == (281, 256, 3):
                    img.resize((434, 384, 3))
                    img = cv.hconcat([img, img])
                    img = img[:384,:,:,].copy()
                else:
                    # img.resize((384, 768, 3))
                    _img1 = np.random.randint(0, 1, (384, 768, 3))
                    _l1, _l2 = min([384, img.shape[0]]), min([768, img.shape[1]])
                    _img1[:_l1,:_l2,:,] = img[:_l1,:_l2,:,]
                    img = _img1.copy()
                
                _P[_k] = img
                warnings.warn(f"{getDT()} _P.SET {_k}")
            else:
                time.sleep(0.618)
                img = _P[_k]
                warnings.warn(f"{getDT()} _P.GET {_k}")

        if _task == 2:
            if _k not in _M:
                _x1, _x2 = jDataset(img)
                _y = encoder(_x1, _x2).cpu().numpy()[0]

                _result = "我的眼睛 "
                for _n in range(yN):
                    _m = COLS[_n]
                    _max = _k2v_num[_m] - 1
                    _st, _ed = 32*_n, 32*(_n+1)

                    _yn = np.argmax(_y[_st:_ed], axis=0)
                    _yn = np.where(_yn > _max, _max, _yn).item()
                    _result += f"{_m} {min(_y[_st:_ed][_yn], 0.9999):.2%}可能是 {_v2k[_m][_yn]}, "
                
                _M[_k] = _result
                warnings.warn(f"{getDT()} _M.SET {_k}")
            else:
                time.sleep(0.618)
                _result = _M[_k]
                warnings.warn(f"{getDT()} _M.GET {_k}")

        if _task == 3:
            _NC["C"] += 1
            warnings.warn(f"{getDT()} JOB.COUNT {_NC['C']:08d}")
    
    return threshold(img), f"{time.time()}|{_result}", f"{_ING1}\n{_ING2}"



# 读取图片文件并转换为 Base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


def ferniebot(_i, _k):
    # 临时保存 并进行压缩处理
    cv.imwrite(f"{wrkp}/working.jpg", cv.resize(_i, (24, 24)))
    _i = encode_image(f"{wrkp}/working.jpg")

    _k = _k.split("|")[1]

    if _k not in _E:
        try:
            response = client.chat.completions.create(
                model="ernie-5.0-thinking-preview",
                messages=iMessages(_i, _k),

                stream=False,
                extra_body={
                    "web_search": {
                        "enable": True
                    }
                },
                max_completion_tokens=65536,
            )
            _result = response.choices[0].message.content
        except:
            _result = "Sorry."
        
        _E[_k] = _result
        warnings.warn(f"{getDT()} _E.SET {_k[:50]}")
    else:
        time.sleep(3.618 + random.random())
        _result = _E[_k]
        warnings.warn(f"{getDT()} _E.GET {_k[:50]}")
    
    return _ENDS, _result


with gr.Blocks(
        title="EDA", 
        theme=gr.themes.Glass(), # Default/Soft/Monochrome/Glass
    ) as APP:

    with gr.Tab("EDA"):
        with gr.Column():
            m1 = gr.Markdown(_HEAD)
            i1 = gr.Image(
                show_label=False, 
                sources=["upload"],
            )
            eg = gr.Examples(
                examples=[f"{wrkp}/{i}.jpg" for i in range(4)], # 16
                label="请上传或选择一张FFA/眼底萤光血管造影:",
                inputs=i1, 
                outputs=i1,
                fn=lambda x: x,
                run_on_click=True,
                examples_per_page=4,
            )
        t1 = gr.Textbox(visible=False)
        b1 = gr.Button(value="开始诊断", size="sm")
        with gr.Row(variant="panel"):
            m4 = gr.Markdown()

        b1.click(inputs=i1, outputs=[i1, t1, m1], fn=fmodel)
        t1.change(inputs=[i1, t1], outputs=[m1, m4], fn=ferniebot)


    with gr.Tab("About"):
        gr.Markdown(
        f"""
        ### EDA(Eye Disease Diagnostic AI Assistant)
        """)
        with gr.Row(variant="panel"):
            gr.Markdown(
            f"""
            * 您只需要上传一张FFA/眼底萤光血管造影，便可以获得清晰的诊断分析，让EDA眼部疾病诊断AI助手成为你的线上诊断专家。只需一键，就能得到专业的眼部疾病诊断，就像按下快门捕捉瞬间一样简单，让您的眼部健康问题得到迅速解答。这款应用不仅能够利用最新的医疗图像识别技术，为你提供快速、准确的眼部疾病诊断，还能够为你提供深入的疾病分析和个性化的治疗建议。
            """)
        url1 = '<a href="https://aistudio.baidu.com/personalcenter/thirdview/979775">Baidu</a>'
        url2 = '<a href="https://github.com/IvanaXu">GitHub</a>'
        url3 = '<a href="mailto:eda@tangjt.cn">eda@tangjt.cn</a>'
        gr.Markdown(
        f"""        
        THANKS ALL !
        <p align="right"><small>{url1} / {url2} , {url3}</small></p>
        <p align="right"><small>{UUID}-030</small></p>
        """)


APP.queue()
APP.launch(
    # server_name="0.0.0.0", 
    # server_port=7860,
    # ssl_keyfile="./data/ssl/key.pem",
    # ssl_certfile="./data/ssl/cert.pem",
)



## 写在最后

您只需要上传一张FFA/眼底萤光血管造影，便可以获得清晰的诊断分析，让EDA眼部疾病诊断AI助手成为你的线上诊断专家。只需一键，就能得到专业的眼部疾病诊断，就像按下快门捕捉瞬间一样简单，让您的眼部健康问题得到迅速解答。这款应用不仅能够利用最新的医疗图像识别技术，为你提供快速、准确的眼部疾病诊断，还能够为你提供深入的疾病分析和个性化的治疗建议。

以上仅为Demo，正式版应用和后续迭代，相信在不久之后会与大家再会，尽请期待。

更多的，欢迎关注[AI Studio](https://aistudio.baidu.com/personalcenter/thirdview/979775)/[GitHub](https://github.com/IvanaXu)。
